링크 : https://www.kaggle.com/code/khushi00452/restaurant-revenue-prediction-challenge

# 1. 주제 : Restaurant Revenue Prediction
**: Open Date, City, City Group, Type, P1~P37 변수들을 활용하여 새로운 레스토랑의 최종 수익(revenue)을 예측하는 회귀(Regression) 문제**

# 2. 데이터
**- train.csv**
- Id: 샘플별 고유 ID
- revenue: 타겟 변수. 레스토랑의 최종 수익
- 익명 처리된 특징 데이터(37개): P1~P37
- 그 외 특징 데이터(4개): Open Date, City, City Group, Type

**- test.csv**
- Id: 샘플별 고유 ID
- 익명 처리된 특징 데이터(37개): P1~P37
- 그 외 특징 데이터(4개): Open Date, City, City Group, Type

**- sample_submission.csv**
- Id: 샘플별 고유 ID
- Prediction: 예측된 레스토랑 수익(revenue)

# 3. 코드 흐름
**1. 데이터 탐색 및 타겟 변수 변환**
- 기초 통계 및 결측치 확인: info(), describe(), isnull().sum() 함수를 통해 총 137개의 행으로 이루어져 있으며 결측치가 하나도 없는 데이터임을 파악
- 타겟 분포 시각화 및 정규화: sns.histplot()으로 타겟 변수(revenue)가 우Right-skewed 분포임을 확인. 모델 학습의 안정성을 위해 np.log1p()를 적용해 로그 스케일로 변환함. 사분위수(IQR) 방식을 사용해 5개의 이상치 식별

**2. 데이터 전처리 및 파생 변수 생성**
- 날짜 기반 피처 생성: pd.to_datetime()으로 문자열을 날짜형으로 변환 후, pd.Timestamp.today()를 활용해 현재 시점 기준 레스토랑 연차(restaurant_age)를 계산함
- 파생 변수 추가: is_big_city라는 이진 변수를 만들고, P1~P37 변수들의 행별 통계치(sum, mean, max, std)를 추출. 이후 is_big_city * P28 이나 restaurant_age * P17 처럼 기존 변수들을 곱한 상호작용(Interaction) 파생 변수를 새롭게 추가

**3. 모델링 및 하잉퍼파라미터 튜닝**
- 스케일링 및 인코딩: ColumnTransformer를 사용해 수치형 변수는 StandardScaler()로, 범주형 변수는 차원 폭발을 막기 위해 TargetEncoder()를 적용해 전처리
- 단일 모델 훈련 및 과적합 진단: LinearRegression, ElasticNet, RandomForestRegressor, CatBoostRegressor 모델을 학습시킴. 데이터 수(137개)가 적어 선형 회귀에서 Test R²가 음수(-0.47)가 나오는 극심한 과적합 현상을 확인함
- 모델 최적화: 과적합 방지를 위해 KFold로 교차 검증을 세팅하고, RandomizedSearchCV를 통해 CatBoostRegressor의 깊이(depth), 학습률(learning_rate) 등 최적의 하이퍼파라미터를 탐색함

**4. 변수 선택 및 앙상블, 마무리**
- 변수 선택(Feature Selection): mutual_info_regression과 CatBoost의 get_feature_importance()를 사용해 영향력이 높은 상위 피처를 선별함. PolynomialFeatures를 활용해 기계적으로 생성된 2차 상호작용 변수들도 성능 테스트에 활용
- 모델 앙상블: 단일 모델의 한계를 보완하고자 튜닝된 CatBoost 모델과 RandomForest 모델의 예측값을 7:3 비율로 가중 평균하여 최종 예측을 수행
- 스케일 복원 및 제출: 예측값(log_predictions)을 기존의 단위로 되돌리기 위해 np.expm1()을 적용한 후, test 셋의 Id와 결합하여 submission.csv 파일로 저장하며 분석을 마무리

# 3-1. 주요 코드

In [ ]:
# 1. CatBoost 하이퍼파라미터 튜닝

cb = CatBoostRegressor(
    iterations=500,
    learning_rate=0.03,
    depth=4,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

cb.fit(
    X_train_processed,
    y_train
)

In [ ]:
# 2. PolynomialFeatures Interaction 변수 생성

poly = PolynomialFeatures(
    degree=2,
    interaction_only=True,
    include_bias=False)

X_train_interactions = poly.fit_transform(X_train_num)

X_test_interactions = poly.transform(X_test_num)

interaction_feature_names = poly.get_feature_names_out(
    selected_num_features)

# 4. 새롭게 알게 된 내용/ 어려운 내용/ 배울 점
- CatBoost Regressor: 트리 기반의 머신러닝 앙상블 기법 중 하나로, 범주형 데이터를 별도의 복잡한 전처리 없이도 내부적으로 잘 처리해 주는 강력한 알고리즘임을 새롭게 알게 됨
- PolynomialFeatures의 활용: 특성(변수)의 의미를 알 수 없을 때(익명 변수 P1~P37), 주어진 변수들끼리 서로 곱해서(interaction_only=True) 기계적으로 새로운 파생 변수를 만들어내는 수학적 접근 방식. 숨겨진 변수 간의 상호작용를 찾아내는 과정을 학습함
